In [1]:
import torch
import torchvision.models as models
from torchvision import transforms
from torchvision.datasets import ImageFolder
from torch.utils.data import DataLoader
from PIL import Image
import os
from sklearn.metrics import roc_auc_score
import torch.nn.functional as F
import numpy as np
import torch.nn as nn
import torch.optim as optim
import json
from tqdm.notebook import tqdm
import gc

In [2]:
gc.collect()
# Clear PyTorch's resident memory
torch.cuda.empty_cache()

Face verification data loading for AUC and ROC calculation

In [3]:
class FaceVerificationDataset(torch.utils.data.Dataset):
    def __init__(self, txt_file, root_dir, transform=None):
        self.root_dir = root_dir
        self.transform = transform
        self.pairs = []

        with open(txt_file, 'r') as f:
            for line in f:
                parts = line.strip().split()
                if len(parts) == 3:
                    self.pairs.append((parts[0], parts[1], int(parts[2])))

    def __len__(self):
        return len(self.pairs)

    def __getitem__(self, idx):
        img1_path, img2_path, label = self.pairs[idx]

        img1_full_path = os.path.join(self.root_dir, img1_path)
        img2_full_path = os.path.join(self.root_dir, img2_path)

        img1 = Image.open(img1_full_path).convert('RGB')
        img2 = Image.open(img2_full_path).convert('RGB')

        if self.transform:
            img1 = self.transform(img1)
            img2 = self.transform(img2)

        return img1, img2, torch.tensor(label, dtype=torch.float32)

Data loading and augmentation:

In [4]:

transforms = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.RandomHorizontalFlip(), 
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
])

In [5]:
train_dataset = ImageFolder(root='../data/classification_data/train_data', transform=transforms)
train_loader = DataLoader(
    train_dataset, 
    batch_size=64, 
    shuffle=True,    
    num_workers=4,   
    pin_memory=True   
)

In [6]:
class_val_dataset = ImageFolder(
    root='../data/classification_data/val_data', 
    transform=transforms,

)
class_val_loader = DataLoader(
    class_val_dataset, 
    batch_size=64, 
    shuffle=False, 
    num_workers=4
)

verfication loading

In [7]:
verification_root = '../data' 

verification_dataset = FaceVerificationDataset(
    txt_file='../data/verification_pairs_val.txt', 
    root_dir=verification_root, 
    transform= transforms
)

verification_loader = torch.utils.data.DataLoader(verification_dataset, batch_size=32, shuffle=False)

Calculating ROC and AUC

In [8]:
def validate_verification_auc(model, val_loader, device):
    model.eval()
    all_labels = []
    all_cosine_scores = []
    all_euclidean_distances = []

    with torch.no_grad():
        for img1, img2, labels in val_loader:
            img1, img2 = img1.to(device), img2.to(device)

            feat1 = model.features(img1)
            feat1 = model.avgpool(feat1).flatten(1)
            
            feat2 = model.features(img2)
            feat2 = model.avgpool(feat2).flatten(1)

            cos_sim = F.cosine_similarity(feat1, feat2)
            euc_dist = torch.cdist(feat1.unsqueeze(1), feat2.unsqueeze(1)).squeeze()
            
            all_cosine_scores.extend(cos_sim.cpu().numpy())
            all_euclidean_distances.extend(euc_dist.cpu().numpy())
            all_labels.extend(labels.cpu().numpy())
            

    auc_cosine = roc_auc_score(all_labels, all_cosine_scores)
    auc_euclidean = roc_auc_score(all_labels, -np.array(all_euclidean_distances))

    return auc_cosine, auc_euclidean

In [9]:
weights = models.EfficientNet_B0_Weights.DEFAULT
model = models.efficientnet_b0(weights=weights)

for name, param in model.named_parameters():
    print(f"Layer: {name} | Size: {param.size()}")


Layer: features.0.0.weight | Size: torch.Size([32, 3, 3, 3])
Layer: features.0.1.weight | Size: torch.Size([32])
Layer: features.0.1.bias | Size: torch.Size([32])
Layer: features.1.0.block.0.0.weight | Size: torch.Size([32, 1, 3, 3])
Layer: features.1.0.block.0.1.weight | Size: torch.Size([32])
Layer: features.1.0.block.0.1.bias | Size: torch.Size([32])
Layer: features.1.0.block.1.fc1.weight | Size: torch.Size([8, 32, 1, 1])
Layer: features.1.0.block.1.fc1.bias | Size: torch.Size([8])
Layer: features.1.0.block.1.fc2.weight | Size: torch.Size([32, 8, 1, 1])
Layer: features.1.0.block.1.fc2.bias | Size: torch.Size([32])
Layer: features.1.0.block.2.0.weight | Size: torch.Size([16, 32, 1, 1])
Layer: features.1.0.block.2.1.weight | Size: torch.Size([16])
Layer: features.1.0.block.2.1.bias | Size: torch.Size([16])
Layer: features.2.0.block.0.0.weight | Size: torch.Size([96, 16, 1, 1])
Layer: features.2.0.block.0.1.weight | Size: torch.Size([96])
Layer: features.2.0.block.0.1.bias | Size: torc

In [10]:
num_classes = len(train_dataset.classes)

for param in model.parameters():
    param.requires_grad = False

for param in model.features[7].parameters():
    param.requires_grad = True

for param in model.features[6].parameters():
    param.requires_grad = True

for param in model.features[5].parameters():
    param.requires_grad = True

for param in model.features[4].parameters():
    param.requires_grad = True

num_ftrs = model.classifier[1].in_features
model.classifier[1] = torch.nn.Linear(num_ftrs, num_classes)

optimizer =optim.Adam([
    {'params': model.features[4].parameters(), 'lr': 2e-5},
    {'params': model.features[5].parameters(), 'lr': 2e-5},
    {'params': model.features[6].parameters(), 'lr': 2e-5},
    {'params': model.features[7].parameters(), 'lr': 2e-5},
    {'params': model.classifier[1].parameters(), 'lr': 2e-4}])

scheduler = optim.lr_scheduler.ReduceLROnPlateau(optimizer, mode='max', factor=0.1, patience=2)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model.to(device)

criterion = nn.CrossEntropyLoss()

In [11]:
print(f"Is CUDA available? {torch.cuda.is_available()}")
print(f"CUDA version: {torch.version.cuda}")
print(f"Device Count: {torch.cuda.device_count()}")

Is CUDA available? True
CUDA version: 13.0
Device Count: 1


In [12]:
num_epochs = 30
best_auc = 0.0
history = {
    'train_loss': [],
    'train_acc': [],
    'val_loss':[],
    'val_acc':[],
    'cos_auc': [],
    'euc_auc': []
}

for epoch in range(num_epochs):
    model.train()
    running_loss = 0.0
    correct_preds = 0
    total_preds = 0

    loop = tqdm(train_loader, desc=f"Epoch [{epoch+1}/{num_epochs}]", leave=True)

    # --- TRAINING PHASE ---
    for images, labels in loop:
        images, labels = images.to(device), labels.to(device)

        optimizer.zero_grad()
        
        outputs = model(images)
        loss = criterion(outputs, labels)
        
        loss.backward()
        optimizer.step()

        running_loss += loss.item() * images.size(0)
        _, predicted = torch.max(outputs, 1)
        total_preds += labels.size(0)
        correct_preds += (predicted == labels).sum().item()
        loop.set_postfix(loss=loss.item(), acc=correct_preds/total_preds)

    model.eval()
    val_loss = 0.0
    val_correct = 0
    val_total = 0

    with torch.no_grad():
        for images, labels in class_val_loader:
            images, labels = images.to(device), labels.to(device)
            outputs = model(images)
            
            # Calculate loss and accuracy
            loss = criterion(outputs, labels)
            val_loss += loss.item() * images.size(0)
            
            _, predicted = torch.max(outputs, 1)
            val_total += labels.size(0)
            val_correct += (predicted == labels).sum().item()

    epoch_loss = running_loss / len(train_dataset)
    epoch_acc = correct_preds / total_preds
    epoch_val_loss = val_loss / len(class_val_dataset)
    epoch_val_acc = val_correct / val_total


    cos_auc, euc_auc = validate_verification_auc(model, verification_loader, device)

    history['train_loss'].append(epoch_loss)
    history['train_acc'].append(epoch_acc)
    history['val_loss'].append(epoch_val_loss)
    history['val_acc'].append(epoch_val_acc)
    history['cos_auc'].append(cos_auc)
    history['euc_auc'].append(euc_auc)

    print(f"Epoch [{epoch+1}/{num_epochs}]")
    print(f"Train Loss: {epoch_loss:.4f} | Train Acc: {epoch_acc:.4f}")
    print(f"Val Loss: {epoch_val_loss:.4f} | Train Acc: {epoch_val_acc:.4f}")
    print(f"Cosine AUC: {cos_auc:.4f} | Euclidean AUC: {euc_auc:.4f}")
    print("-" * 30)

    if cos_auc > best_auc:
        best_auc = cos_auc
        torch.save(model.state_dict(), 'best_supervised_model.pth')
        print("Model saved based on Cosine AUC!")

    scheduler.step(cos_auc)


# Save history to a JSON file
with open('training_history_supervised.json', 'w+') as f:
    json.dump(history, f)

Epoch [1/30]:   0%|          | 0/5948 [00:00<?, ?it/s]

Epoch [1/30]
Train Loss: 5.7714 | Train Acc: 0.1242
Val Loss: 3.8756 | Train Acc: 0.2915
Cosine AUC: 0.8453 | Euclidean AUC: 0.8750
------------------------------
Model saved based on Cosine AUC!


Epoch [2/30]:   0%|          | 0/5948 [00:00<?, ?it/s]

Epoch [2/30]
Train Loss: 3.1519 | Train Acc: 0.4017
Val Loss: 2.6760 | Train Acc: 0.4796
Cosine AUC: 0.8589 | Euclidean AUC: 0.8824
------------------------------
Model saved based on Cosine AUC!


Epoch [3/30]:   0%|          | 0/5948 [00:00<?, ?it/s]

...

Epoch [28/30]
Train Loss: 0.0888 | Train Acc: 0.9799
Val Loss: 1.1621 | Train Acc: 0.7874
Cosine AUC: 0.9026 | Euclidean AUC: 0.6658
------------------------------
Model saved based on Cosine AUC!


Epoch [29/30]:   0%|          | 0/5948 [00:00<?, ?it/s]

Epoch [29/30]
Train Loss: 0.0865 | Train Acc: 0.9805
Val Loss: 1.1547 | Train Acc: 0.7895
Cosine AUC: 0.9013 | Euclidean AUC: 0.6567
------------------------------


Epoch [30/30]:   0%|          | 0/5948 [00:00<?, ?it/s]

Epoch [30/30]
Train Loss: 0.0856 | Train Acc: 0.9807
Val Loss: 1.1626 | Train Acc: 0.7900
Cosine AUC: 0.9008 | Euclidean AUC: 0.6573
------------------------------
